In [1]:
import importlib.util
import subprocess


def detect_gpu() -> dict:
    """Detect whether a GPU is visible and whether a Python GPU backend is usable."""
    info = {
        "gpu_visible": False,
        "backend": None,
        "details": "No GPU detected.",
    }

    if importlib.util.find_spec("torch") is not None:
        import torch

        if torch.cuda.is_available():
            info["gpu_visible"] = True
            info["backend"] = "torch"
            info["details"] = (
                f"PyTorch CUDA available with {torch.cuda.device_count()} device(s): "
                + ", ".join(torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count()))
            )
            return info

    if importlib.util.find_spec("tensorflow") is not None:
        import tensorflow as tf

        gpus = tf.config.list_physical_devices("GPU")
        if gpus:
            info["gpu_visible"] = True
            info["backend"] = "tensorflow"
            info["details"] = f"TensorFlow detected {len(gpus)} GPU device(s)."
            return info

    try:
        probe = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True,
            text=True,
            check=False,
        )
        gpu_names = [line.strip() for line in probe.stdout.splitlines() if line.strip()]
        if probe.returncode == 0 and gpu_names:
            info["gpu_visible"] = True
            info["backend"] = "nvidia-smi"
            info["details"] = "NVIDIA GPU visible: " + ", ".join(gpu_names)
    except FileNotFoundError:
        pass

    return info


gpu_status = detect_gpu()
USE_GPU = gpu_status["gpu_visible"] and gpu_status["backend"] in {"torch", "tensorflow"}

print(gpu_status["details"])
print(f"USE_GPU={USE_GPU}")

if gpu_status["gpu_visible"]:
    print(
        "Note: The current `mlp_pipeline` uses scikit-learn MLP, which trains on CPU. "
        "GPU detection is exposed for environment checks and future GPU-enabled model paths."
    )

No GPU detected.
USE_GPU=False


# Neural Authorship Attribution with `mlp_pipeline`

This notebook reproduces a full Hamilton vs Madison authorship experiment using the reusable pipeline in `lexos.classification.mlp_pipeline`.

It trains on known papers (**HAMILTON**, **MADISON**), evaluates holdout and cross-validation performance, and predicts likely authorship for unknown papers (**DISPUTED**, **COAUTHORED**).

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from lexos.classification.mlp_pipeline import (
    MLPPipelineConfig,
    run_mlp_authorship_pipeline,
    save_mlp_unknown_predictions,
)

## Locate Data and Build Datasets

We locate the `fed_papers` directory, separate known-author papers for training, and collect unknown papers for inference.

In [3]:
SEED = 42
np.random.seed(SEED)

base = Path.cwd()
search_roots = [base] + list(base.parents)

data_dir = next((root / "fed_papers" for root in search_roots if (root / "fed_papers").exists()), None)
if data_dir is None:
    raise FileNotFoundError("Could not locate 'fed_papers' from the current notebook location.")

train_dirs = ["HAMILTON", "MADISON"]
unknown_dirs = ["DISPUTED", "COAUTHORED"]

train_files = []
train_labels = []
for author in train_dirs:
    files = sorted((data_dir / author).glob("*.txt"))
    train_files.extend(files)
    train_labels.extend([author] * len(files))

unknown_files = []
unknown_sets = []
for subset in unknown_dirs:
    files = sorted((data_dir / subset).glob("*.txt"))
    unknown_files.extend(files)
    unknown_sets.extend([subset] * len(files))

train_texts = [p.read_text(encoding="utf-8", errors="ignore") for p in train_files]
unknown_texts = [p.read_text(encoding="utf-8", errors="ignore") for p in unknown_files]
unknown_ids = [p.name for p in unknown_files]

print("Using data directory:", data_dir)
print(f"Training docs: {len(train_texts)}")
print(f"Unknown docs: {len(unknown_texts)}")
print(pd.Series(train_labels).value_counts())

Using data directory: c:\Users\lakot\Downloads\lexos\doc_src\docs\tutorials\classification\fed_papers
Training docs: 65
Unknown docs: 15
HAMILTON    51
MADISON     14
Name: count, dtype: int64


## Configure and Run Pipeline

The pipeline performs leakage-safe preprocessing, holdout evaluation, cross-validation, final refit, and optional inference on unknown documents.

In [4]:
if USE_GPU:
    print("GPU detected; current mlp_pipeline path remains CPU-bound (scikit-learn MLP backend).")
else:
    print("No compatible GPU backend detected. Running pipeline on CPU.")

cfg = MLPPipelineConfig(
    seed=SEED,
    min_df=2,
    test_size=0.2,
    cv_splits=5,
    include_bigrams=True,
    use_smote=True,
    mlp_kwargs={
        "hidden_layer_sizes": (64,),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-4,
        "learning_rate_init": 1e-3,
        "max_iter": 1000,
    },
)

results = run_mlp_authorship_pipeline(
    train_data=train_texts,
    train_labels=train_labels,
    test_data=unknown_texts,
    test_ids=unknown_ids,
    config=cfg,
)

No compatible GPU backend detected. Running pipeline on CPU.


c:\Users\lakot\Downloads\lexos\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\lakot\Downloads\lexos\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\lakot\AppData\Roaming\uv\python\cpython-3.12.3-windows-x86_64-none\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lakot\AppData\Roaming\uv\python\cpython-3.12.3-windows-x86_64-none\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_f

## Holdout Evaluation

Metrics below come from the leakage-safe holdout split generated inside the pipeline.

In [5]:
print("Holdout metrics:")
print(results.holdout_metrics)

print("\nClassification report:")
results.holdout_report

Holdout metrics:
{'accuracy': 0.9230769230769231, 'balanced_accuracy': 0.95, 'macro_f1': 0.9022556390977443}

Classification report:


,precision,recall,f1-score,support
HAMILTON,1.000000,0.900000,0.947368,10.000000
MADISON,0.750000,1.000000,0.857143,3.000000
accuracy,0.923077,0.923077,0.923077,0.923077
macro avg,0.875000,0.950000,0.902256,13.000000
weighted avg,0.942308,0.923077,0.926547,13.000000


In [6]:
print("Holdout confusion matrix:")
results.holdout_confusion_matrix

Holdout confusion matrix:


,pred_HAMILTON,pred_MADISON
true_HAMILTON,9,1
true_MADISON,0,3


## Cross-Validation Metrics

These values summarize the leakage-safe stratified CV executed inside the pipeline.

In [7]:
print("Per-fold CV metrics:")
results.cv_fold_metrics

print("\nMean CV metrics:")
results.cv_mean_metrics

Per-fold CV metrics:

Mean CV metrics:


{'accuracy': 0.8615384615384617,
 'balanced_accuracy': 0.7533333333333333,
 'macro_f1': 0.7392040610710688}

## Inference on Unknown Papers

Predictions below are generated by the final model refit on all known labeled papers.

In [8]:
results_with_sets = results.test_predictions.merge(
    pd.DataFrame({"sample_id": unknown_ids, "set": unknown_sets}),
    on="sample_id",
    how="left",
)

results_with_sets = results_with_sets[["sample_id", "set", "predicted_label"] + [
    col for col in results_with_sets.columns if col.startswith("p_")
]]

results_with_sets.sort_values(["set", "sample_id"]).reset_index(drop=True)

,sample_id,set,predicted_label,p_hamilton,p_madison
0,FED_18_C.txt,COAUTHORED,MADISON,0.052470,0.947530
1,FED_19_C.txt,COAUTHORED,MADISON,0.157986,0.842014
2,FED_20_C.txt,COAUTHORED,MADISON,0.415020,0.584980
3,FED_49_D.txt,DISPUTED,MADISON,0.022633,0.977367
4,FED_50_D.txt,DISPUTED,HAMILTON,0.745548,0.254452
5,FED_51_D.txt,DISPUTED,MADISON,0.000194,0.999806
6,FED_52_D.txt,DISPUTED,MADISON,0.019389,0.980611
7,FED_53_D.txt,DISPUTED,MADISON,0.021899,0.978101
8,FED_54_D.txt,DISPUTED,MADISON,0.002919,0.997081
9,FED_55_D.txt,DISPUTED,MADISON,0.009568,0.990432


## Save Predictions

Export unknown-paper predictions to CSV for downstream analysis.

In [9]:
output_csv = data_dir / "neural_authorship_predictions_pipeline.csv"
saved_path = save_mlp_unknown_predictions(results, output_csv)
print("Saved predictions:", saved_path)

Saved predictions: C:\Users\lakot\Downloads\lexos\doc_src\docs\tutorials\classification\fed_papers\neural_authorship_predictions_pipeline.csv
